# CulturalQA RAG — All 7 Datasets, Two Answer Paths, 5/7 Verified

Two answer paths:
- **Path A (exact lookup):** "what is artifact #412 in DOSA?" -> answered by direct
  dictionary lookup over the real dataset rows, no search involved, deterministic.
- **Path B (RAG):** "how many artifacts does DOSA have?" / "which artifacts relate to
  weddings?" -> full pipeline: FAISS+BM25 hybrid retrieval -> MultiQuery expansion ->
  Cross-Encoder rerank -> BERT+CLIP encoding -> MoE fusion re-score -> extractive reader.

**Verified against real data (5 of 7):**
- DOSA and SCALE pulled live from their GitHub repos.
- DIWALI, DRISHTIKON, and Indian Cultural Dataset confirmed via their HuggingFace
  dataset viewer pages (all three are ungated -- no login needed).
- GIMMICK remains gated (confirmed: "You need to agree to share your contact
  information to access this dataset") -- needs your HF login + license acceptance.
- Durga QA is on Mendeley Data, not checked yet.

See Section 12 for exact one-line commands to pull the ungated ones yourself.

**Setup before running:**
1. Put EDA report PDFs in `./data/` (for Path B conceptual/stats questions).
2. Put real dataset files in `./data/raw/<Dataset>/...` (for Path A exact lookup +
   Path B content search) -- folder layout shown in Section 1.
3. Set `GROQ_API_KEY` (used only for cheap query expansion).
4. Run top to bottom.


## 0. Install dependencies

In [ ]:
%%capture
!pip install langchain langchain-community langchain-groq faiss-cpu sentence-transformers \
    rank_bm25 pypdf unstructured openpyxl jq transformers torch pillow requests nbformat

## 0.1 Set your Groq API key

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "your_key_here"  # <-- replace this

## 0.5 Shared dataset config

Lightweight dataset name/alias lookup, kept separate from the heavy file loaders so
that checking "did they mention DOSA in this question" doesn't require importing
PDF/Excel parsers.

In [ ]:
"""
dataset_config.py
------------------
Small shared constants used by both ingestion.py (heavy, pulls in
LangChain loaders) and query_router.py (needs to stay lightweight since
it runs on every single question). Keeping this separate means asking a
question doesn't have to import PDF/Excel/CSV loaders just to check
"did they type 'DOSA' in their question."
"""

DATASET_ALIASES = {
    "gimmick": "GIMMICK",
    "scale": "SCALE",
    "drishtikon": "DRISHTIKON",
    "diwali": "DIWALI",
    "dosa": "DOSA",
    "durga": "Durga QA",
    "ombhojane": "Indian Cultural Dataset",
}

# Which datasets can carry a real per-record image field (per each report's
# Modality/Image Statistics section). Used by encoders.py so CLIP only runs
# where images actually exist.
HAS_IMAGE_MODALITY = {
    "GIMMICK": True,
    "DOSA": True,
    "DRISHTIKON": True,
    "SCALE": False,
    "DIWALI": False,
    "Durga QA": False,
    "Indian Cultural Dataset": False,
}


## 1. Ingestion -- multi-source loading (Reference #1)

Loads the EDA report PDFs (for Path B conceptual questions). Required folder layout
for the *real* dataset files (Path A + richer Path B) is:

```
./data/raw/DOSA/<state>/original_artifacts.csv
./data/raw/DOSA/<state>/expanded_artifacts.csv
./data/raw/SCALE/cuisine.xlsx, landmarks.xlsx, ... (7 files)
./data/raw/GIMMICK/...
./data/raw/DRISHTIKON/...
./data/raw/DIWALI/...
./data/raw/Durga QA/...
./data/raw/Indian Cultural Dataset/...
```

In [ ]:
"""
ingestion.py
------------
Multi-source loading for all 7 datasets (GIMMICK, SCALE, DRISHTIKON, DIWALI,
DOSA, Durga QA, Indian Cultural Dataset / ombhojane).

Follows reference #1 (multi-source LangChain RAG): one loader per file type,
unified into a single corpus with per-chunk metadata identifying which
dataset a fact came from.

Follows reference #2 (RAG with custom unstructured data): report PDFs mix
prose/tables/captions -> we don't try to parse tables structurally here,
we lean on chunking (see chunking.py) to keep table-adjacent text intact.
Structured sources (csv/json/xlsx of the *actual* dataset records, not just
the EDA reports) are loaded row-wise so image/text fields stay paired.

Only 3 of the 7 datasets carry real per-record images (per their EDA
reports): DOSA (optional per-artifact image), DRISHTIKON (image URL per
MCQA instance), GIMMICK (CIVQA/QA-base image bytes/URLs). The other four
(SCALE, DIWALI, Durga QA, Indian Cultural Dataset) are text-only by design.
This is encoded in HAS_IMAGE_MODALITY below and drives whether the
multimodal encoder (encoders.py) bothers running CLIP on a given record.
"""

import os
import glob

from langchain_community.document_loaders import (
    PyPDFLoader,
    CSVLoader,
    JSONLoader,
    UnstructuredExcelLoader,
    TextLoader,
)

from dataset_config import DATASET_ALIASES, HAS_IMAGE_MODALITY

# Column names across the 7 datasets' actual data files (not the EDA
# reports) that hold an image reference, per each dataset's field docs.
IMAGE_FIELD_CANDIDATES = [
    "ich_image", "ich_image_url", "ich_image_urls",   # GIMMICK
    "image_url", "image_path", "images",               # DOSA / generic
    "video_center_frame_fn",                            # GIMMICK VVQA (frame)
]


def infer_dataset_tag(filepath: str) -> str:
    name = os.path.basename(filepath).lower()
    for key, label in DATASET_ALIASES.items():
        if key in name:
            return label
    return os.path.splitext(os.path.basename(filepath))[0]


def _attach_image_metadata(doc, tag: str):
    """If this row/doc has an image field and the dataset supports images,
    tag it so the multimodal encoder knows to run CLIP on it."""
    has_image_field = any(
        field in doc.metadata or field in doc.page_content.lower()
        for field in IMAGE_FIELD_CANDIDATES
    )
    doc.metadata["has_image"] = bool(HAS_IMAGE_MODALITY.get(tag, False) and has_image_field)
    return doc


def load_all_documents(data_dir: str = "./data"):
    """Loads every supported file in data_dir. Report PDFs (EDA writeups)
    and structured record files (csv/json/xlsx of actual dataset rows) can
    both live in the same folder -- they're distinguished by extension,
    not by content, matching the multi-loader pattern from reference #1."""
    docs = []
    patterns_loaders = {
        "*.pdf": lambda p: PyPDFLoader(p).load(),
        "*.csv": lambda p: CSVLoader(p).load(),
        "*.json": lambda p: JSONLoader(p, jq_schema=".", text_content=False).load(),
        "*.xlsx": lambda p: UnstructuredExcelLoader(p, mode="elements").load(),
        "*.txt": lambda p: TextLoader(p, encoding="utf-8").load(),
        "*.md": lambda p: TextLoader(p, encoding="utf-8").load(),
    }

    for pattern, loader_fn in patterns_loaders.items():
        for filepath in glob.glob(os.path.join(data_dir, pattern)):
            try:
                loaded = loader_fn(filepath)
            except Exception as e:
                print(f"[skip] {filepath}: {e}")
                continue
            tag = infer_dataset_tag(filepath)
            for d in loaded:
                d.metadata["dataset"] = tag
                d.metadata["source_file"] = os.path.basename(filepath)
                _attach_image_metadata(d, tag)
            docs.extend(loaded)
            print(f"[loaded] {filepath} -> {len(loaded)} docs ({tag}, "
                  f"image-capable={HAS_IMAGE_MODALITY.get(tag, False)})")

    if not docs:
        print(f"No files found in {data_dir}.")
    return docs


## 2. Structured Record Store -- the fix for exact-lookup questions (NEW)

Semantic chunking + embedding search can only answer questions about *descriptions*
of a dataset. "What's artifact #412?" has one correct answer that either exists or
doesn't -- the wrong job for "find the most semantically similar paragraph."

This loads the REAL dataset files row by row and builds an in-memory
`(dataset, id) -> record` index for O(1) exact lookup. Verified against real DOSA
(`artifact`, `clues`) and SCALE (`Item`, `Translation`) schemas pulled live from
GitHub. Neither dataset has a built-in numeric ID in its source files -- IDs are
assigned deterministically (sorted folder/file order, then row order) so lookups are
reproducible.

In [ ]:
"""
structured_store.py
--------------------
This is the actual fix for "what's artifact #412 in DOSA?" style questions.

Semantic RAG (chunking + embedding search) answers questions about
*descriptions* of a dataset ("how many artifacts does DOSA have"). It
cannot answer "what is record #412", because no paragraph anywhere says
that -- that fact only exists as one row in the dataset's real data files,
which the report-based pipeline never loaded.

This module loads the ACTUAL dataset files (not the EDA report PDFs) row
by row, assigns each row a stable ID, and keeps an in-memory index for
O(1) exact lookup -- bypassing embeddings/reranking/extraction entirely
for ID-based questions, which also makes those answers 100% accurate
instead of "most likely chunk."

Verified against real data -- pulled directly from source (GitHub repos
reachable from this environment, or HuggingFace dataset viewer pages
readable via web_fetch even though direct file download isn't):
  - DOSA:  github.com/microsoft/DOSA -> data/<state>/{original,expanded}_artifacts.csv
           columns: artifact, clues (no built-in numeric ID)
  - SCALE: github.com/google-research-datasets/SCALE-Cultural-Data -> data/*.xlsx
           columns: Item, Translation (domain = filename, no built-in ID)
  - DIWALI: huggingface.co/datasets/nlip/DIWALI (ungated, 8,817 rows)
           columns: facet, state, concept, description, source
  - DRISHTIKON: huggingface.co/datasets/13ari/DRISHTIKON (ungated, 64.3k rows)
           columns: language, state, attribute, question_type, question,
                    option1, option2, option3, option4, answer, image_name,
                    image_link, image
  - Indian Cultural Dataset: huggingface.co/datasets/ombhojane/indian-cultural-dataset
           (ungated, 96 rows) columns: text, category -- confirmed matches
           the EDA report exactly. NOTE: some rows run up to ~15.4M
           characters (full Mahabharata text) -- these get chunked even in
           the "raw" path, unlike other datasets' rows, or they'd blow out
           context on a single retrieval hit.

NOT verified against real files -- generic loader will work once the real
files are dropped in, but confirm column names first:
  - GIMMICK (HuggingFace, gated -- requires login + accepting the license,
    confirmed via direct fetch: "You need to agree to share your contact
    information to access this dataset")
  - Durga QA (Mendeley Data, DOI 10.17632/6hcx53kywr.1 -- not checked)

Expected folder layout:
  ./data/raw/DOSA/<state>/original_artifacts.csv
  ./data/raw/DOSA/<state>/expanded_artifacts.csv
  ./data/raw/SCALE/cuisine.xlsx, landmarks.xlsx, ... (7 files)
  ./data/raw/DIWALI/*.csv or *.json  (from `datasets.load_dataset("nlip/DIWALI")`)
  ./data/raw/DRISHTIKON/*.csv or *.parquet  (from `datasets.load_dataset("13ari/DRISHTIKON")`)
  ./data/raw/Indian Cultural Dataset/*.json
  ./data/raw/GIMMICK/*.parquet or *.csv  (requires HF login + license accept)
  ./data/raw/Durga QA/*.csv
"""

import os
import csv
import json
import glob

try:
    import openpyxl
except ImportError:
    openpyxl = None

try:
    import pandas as pd
except ImportError:
    pd = None

from langchain.schema import Document


# ---------------------------------------------------------------------------
# Dataset-specific loaders (real schema where verified)
# ---------------------------------------------------------------------------
def load_dosa_raw(base_dir: str):
    """Real DOSA layout: one folder per state, each with original_artifacts.csv
    and expanded_artifacts.csv, columns = artifact, clues. No numeric ID exists
    in the source data, so we assign one deterministically: sorted state
    folders, original before expanded, row order within file."""
    records = []
    idx = 0
    if not os.path.isdir(base_dir):
        return records

    for state_dir in sorted(os.listdir(base_dir)):
        state_path = os.path.join(base_dir, state_dir)
        if not os.path.isdir(state_path):
            continue
        for fname, split in [("original_artifacts.csv", "original"), ("expanded_artifacts.csv", "expanded")]:
            fpath = os.path.join(state_path, fname)
            if not os.path.exists(fpath):
                continue
            with open(fpath, encoding="utf-8-sig") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    idx += 1
                    records.append({
                        "dataset": "DOSA",
                        "id": idx,
                        "state": state_dir,
                        "split": split,
                        "artifact": row.get("artifact", "").strip(),
                        "clues": row.get("clues", "").strip(),
                    })
    print(f"[structured_store] DOSA: loaded {idx} real artifact records")
    return records


def load_scale_raw(base_dir: str):
    """Real SCALE layout: one xlsx per concept category, columns = Item,
    Translation. Domain comes from the filename, matching the EDA report."""
    records = []
    idx = 0
    if not os.path.isdir(base_dir) or openpyxl is None:
        return records

    for fpath in sorted(glob.glob(os.path.join(base_dir, "*.xlsx"))):
        domain = os.path.splitext(os.path.basename(fpath))[0]
        wb = openpyxl.load_workbook(fpath, read_only=True)
        ws = wb.active
        rows = list(ws.iter_rows(values_only=True))
        if not rows:
            continue
        header = [str(h).strip() for h in rows[0]]
        for r in rows[1:]:
            if not any(r):
                continue
            idx += 1
            rec = dict(zip(header, r))
            rec["dataset"] = "SCALE"
            rec["id"] = idx
            rec["domain"] = domain
            records.append(rec)
    print(f"[structured_store] SCALE: loaded {idx} real item records")
    return records


# ---------------------------------------------------------------------------
# Generic fallback loader for the 5 datasets we couldn't verify live.
# Works on csv/json/xlsx/parquet; assigns a sequential ID unless the file
# already has an obvious id-like column.
# ---------------------------------------------------------------------------
ID_COLUMN_HINTS = ["id", "sample_uuid", "question_id", "artifact_id", "record_id"]


def _rows_from_file(fpath: str):
    ext = os.path.splitext(fpath)[1].lower()
    if ext == ".csv":
        with open(fpath, encoding="utf-8-sig") as f:
            return list(csv.DictReader(f))
    if ext == ".json":
        with open(fpath, encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, list) else [data]
    if ext in (".xlsx", ".xls") and openpyxl is not None:
        wb = openpyxl.load_workbook(fpath, read_only=True)
        ws = wb.active
        rows = list(ws.iter_rows(values_only=True))
        if not rows:
            return []
        header = [str(h).strip() for h in rows[0]]
        return [dict(zip(header, r)) for r in rows[1:] if any(r)]
    if ext == ".parquet" and pd is not None:
        return pd.read_parquet(fpath).to_dict(orient="records")
    return []


def load_generic_raw(dataset_tag: str, base_dir: str):
    records = []
    idx = 0
    if not os.path.isdir(base_dir):
        return records

    for fpath in sorted(glob.glob(os.path.join(base_dir, "*"))):
        rows = _rows_from_file(fpath)
        for row in rows:
            id_col = next((c for c in row if str(c).lower() in ID_COLUMN_HINTS), None)
            idx += 1
            rec = dict(row)
            rec["dataset"] = dataset_tag
            rec["id"] = row.get(id_col, idx) if id_col else idx
            records.append(rec)
    print(f"[structured_store] {dataset_tag}: loaded {idx} generic records from {base_dir}")
    return records


RAW_LOADERS = {
    "DOSA": load_dosa_raw,
    "SCALE": load_scale_raw,
}


# ---------------------------------------------------------------------------
# Verified-schema loaders for the 3 more datasets confirmed via HuggingFace
# dataset viewer (ungated -- readable without login). These read whatever
# CSV/JSON/parquet you've exported/downloaded into ./data/raw/<Dataset>/,
# and just enforce/expect the confirmed column names rather than guessing.
# ---------------------------------------------------------------------------
def load_diwali_raw(base_dir: str):
    """Confirmed real columns (huggingface.co/datasets/nlip/DIWALI):
    facet, state, concept, description, source. No built-in numeric ID."""
    records = []
    idx = 0
    if not os.path.isdir(base_dir):
        return records
    for fpath in sorted(glob.glob(os.path.join(base_dir, "*"))):
        for row in _rows_from_file(fpath):
            idx += 1
            records.append({
                "dataset": "DIWALI",
                "id": idx,
                "facet": row.get("facet", ""),
                "state": row.get("state", ""),
                "concept": row.get("concept", ""),
                "description": row.get("description", ""),
                "source": row.get("source", ""),
            })
    print(f"[structured_store] DIWALI: loaded {idx} real concept records")
    return records


def load_drishtikon_raw(base_dir: str):
    """Confirmed real columns (huggingface.co/datasets/13ari/DRISHTIKON):
    language, state, attribute, question_type, question, option1-4, answer,
    image_name, image_link, image. No built-in numeric ID. image_link is
    kept so encoders.py can run CLIP on it (DRISHTIKON is image-capable)."""
    records = []
    idx = 0
    if not os.path.isdir(base_dir):
        return records
    for fpath in sorted(glob.glob(os.path.join(base_dir, "*"))):
        for row in _rows_from_file(fpath):
            idx += 1
            records.append({
                "dataset": "DRISHTIKON",
                "id": idx,
                "language": row.get("language", ""),
                "state": row.get("state", ""),
                "attribute": row.get("attribute", ""),
                "question_type": row.get("question_type", ""),
                "question": row.get("question", ""),
                "option1": row.get("option1", ""),
                "option2": row.get("option2", ""),
                "option3": row.get("option3", ""),
                "option4": row.get("option4", ""),
                "answer": row.get("answer", ""),
                "image_name": row.get("image_name", ""),
                "image_url": row.get("image_link", ""),  # normalized for encoders.py
            })
    print(f"[structured_store] DRISHTIKON: loaded {idx} real MCQA records")
    return records


# Row text above this length gets its own recursive chunk pass instead of
# being indexed as one giant Document. Indian Cultural Dataset has entries
# up to ~15.4M characters (full Mahabharata text) per its EDA report --
# far too large for a single retrieval hit or an LLM context window.
LARGE_ROW_CHAR_THRESHOLD = 4000


def load_indian_cultural_raw(base_dir: str):
    """Confirmed real columns (huggingface.co/datasets/ombhojane/indian-cultural-dataset):
    text, category. Matches the EDA report exactly. No built-in numeric ID."""
    records = []
    idx = 0
    if not os.path.isdir(base_dir):
        return records
    for fpath in sorted(glob.glob(os.path.join(base_dir, "*"))):
        for row in _rows_from_file(fpath):
            idx += 1
            records.append({
                "dataset": "Indian Cultural Dataset",
                "id": idx,
                "text": row.get("text", ""),
                "category": row.get("category", ""),
            })
    print(f"[structured_store] Indian Cultural Dataset: loaded {idx} real records "
          f"(some may be very large -- see LARGE_ROW_CHAR_THRESHOLD)")
    return records


RAW_LOADERS.update({
    "DIWALI": load_diwali_raw,
    "DRISHTIKON": load_drishtikon_raw,
    "Indian Cultural Dataset": load_indian_cultural_raw,
})


def load_all_raw_records(raw_dir: str = "./data/raw"):
    from dataset_config import DATASET_ALIASES

    all_records = []
    for dataset_tag in set(DATASET_ALIASES.values()):
        ds_dir = os.path.join(raw_dir, dataset_tag)
        if not os.path.isdir(ds_dir):
            print(f"[structured_store] no raw folder for {dataset_tag} at {ds_dir} -- "
                  f"row-level lookup unavailable for this dataset until you add it.")
            continue
        loader = RAW_LOADERS.get(dataset_tag, lambda d: load_generic_raw(dataset_tag, d))
        all_records.extend(loader(ds_dir))
    return all_records


def build_record_index(records):
    """(dataset, id) -> record dict, for O(1) exact lookup."""
    return {(r["dataset"], r["id"]): r for r in records}


def record_to_text(r: dict) -> str:
    if r["dataset"] == "DOSA":
        return f"Artifact: {r['artifact']}. State: {r['state']} ({r['split']}). Description: {r['clues']}"
    if r["dataset"] == "SCALE":
        return f"Item: {r.get('Item')}. Translation: {r.get('Translation')}. Domain: {r.get('domain')}"
    if r["dataset"] == "DIWALI":
        return (f"Concept: {r.get('concept')}. Facet: {r.get('facet')}. State: {r.get('state')}. "
                f"Description: {r.get('description')}")
    if r["dataset"] == "DRISHTIKON":
        return (f"Question ({r.get('language')}, {r.get('state')}, {r.get('attribute')}): "
                f"{r.get('question')} Options: {r.get('option1')} / {r.get('option2')} / "
                f"{r.get('option3')} / {r.get('option4')}. Answer: {r.get('answer')}")
    if r["dataset"] == "Indian Cultural Dataset":
        return f"Category: {r.get('category')}. {r.get('text')}"
    parts = [f"{k}: {v}" for k, v in r.items() if k not in ("dataset", "id") and v not in (None, "")]
    return "; ".join(parts)


def records_to_documents(records):
    """One Document per row, UNCHUNKED (rows are already small) -- these
    feed straight into the same FAISS/BM25 index as the report chunks, so
    conceptual questions ('artifacts related to weddings') can also
    surface real rows, not just report prose.

    Exception: rows whose text exceeds LARGE_ROW_CHAR_THRESHOLD (e.g. Indian
    Cultural Dataset's full-text Mahabharata entries) get split into
    multiple sub-chunks via the same recursive splitter used for report
    text, so a single retrieval hit doesn't try to swallow 15 million
    characters. Each sub-chunk keeps the parent record_id so exact lookup
    (Path A) still resolves to the whole record, while search (Path B)
    only ever returns a manageable slice."""
    from chunking import chunk_documents

    small_docs = []
    large_docs = []
    for r in records:
        text = record_to_text(r)
        meta = {"dataset": r["dataset"], "record_id": r["id"], "source_type": "raw_record"}
        meta.update({k: v for k, v in r.items() if k not in ("dataset", "id")})
        doc = Document(page_content=text, metadata=meta)
        if len(text) > LARGE_ROW_CHAR_THRESHOLD:
            large_docs.append(doc)
        else:
            small_docs.append(doc)

    if large_docs:
        print(f"[structured_store] chunking {len(large_docs)} oversized rows "
              f"(> {LARGE_ROW_CHAR_THRESHOLD} chars) instead of indexing whole")
        large_docs = chunk_documents(large_docs, chunk_size=1200, chunk_overlap=200)

    return small_docs + large_docs


def format_record_answer(record: dict) -> str:
    if record["dataset"] == "DOSA":
        return (f"DOSA artifact #{record['id']} ({record['state']}, {record['split']} set): "
                f"\"{record['artifact']}\" -- {record['clues']}")
    if record["dataset"] == "SCALE":
        return (f"SCALE item #{record['id']} ({record.get('domain')}): "
                f"\"{record.get('Item')}\" (translation: {record.get('Translation')})")
    if record["dataset"] == "DIWALI":
        return (f"DIWALI concept #{record['id']} ({record.get('facet')}, {record.get('state')}): "
                f"\"{record.get('concept')}\" -- {record.get('description')}")
    if record["dataset"] == "DRISHTIKON":
        return (f"DRISHTIKON question #{record['id']} ({record.get('state')}, {record.get('attribute')}, "
                f"{record.get('language')}): {record.get('question')} "
                f"[Options: {record.get('option1')}, {record.get('option2')}, "
                f"{record.get('option3')}, {record.get('option4')}] Answer: {record.get('answer')}")
    if record["dataset"] == "Indian Cultural Dataset":
        preview = (record.get("text") or "")[:300]
        suffix = "..." if len(record.get("text") or "") > 300 else ""
        return f"Indian Cultural Dataset record #{record['id']} ({record.get('category')}): {preview}{suffix}"
    parts = [f"{k}={v}" for k, v in record.items() if k not in ("dataset", "id")]
    return f"{record['dataset']} record #{record['id']}: " + ", ".join(parts)


## 3. Query Router -- deciding Path A vs Path B (NEW)

Detects "artifact #412", "record 7", "item #50 in scale" style questions and routes
them to the exact-lookup path. Everything else (statistics, conceptual, "which
artifacts relate to X") falls through to the RAG pipeline unchanged.

In [ ]:
"""
query_router.py
----------------
Detects "give me record #N" style questions and routes them to a direct
dict lookup in structured_store's record index instead of semantic search.

This matters because embeddings/reranking/extraction are all "find the
most likely relevant text" tools -- they're the wrong tool for "return
exactly row 412," which has one correct answer that either exists or
doesn't. Routing ID questions around the RAG path makes those answers
deterministic instead of "probably right."

Everything that ISN'T an ID-lookup question falls through to the normal
hybrid retrieval + MoE + extractive reader pipeline, unchanged.
"""

import re
from dataset_config import DATASET_ALIASES

LOOKUP_TRIGGERS = [
    "artifact #", "artifact number", "artifact id",
    "record #", "record number", "record id",
    "row #", "row number",
    "item #", "item number", "item id",
    "sample #", "sample number",
    "index ",
]

NUMBER_PATTERN = re.compile(r"#?\s*(\d+)\b")

# catches "record 7", "artifact 412", "item 3" etc. even without a '#' or
# the word "number"/"id" attached
KEYWORD_PLUS_NUMBER = re.compile(
    r"\b(artifact|record|row|item|sample)\s*#?\s*\d+\b", re.IGNORECASE
)


def detect_dataset(question: str):
    q = question.lower()
    for key, label in DATASET_ALIASES.items():
        if key in q:
            return label
    return None


def is_lookup_query(question: str) -> bool:
    q = question.lower()
    has_trigger = (
        any(t in q for t in LOOKUP_TRIGGERS)
        or bool(re.search(r"#\s*\d+", question))
        or bool(KEYWORD_PLUS_NUMBER.search(question))
    )
    has_number = bool(NUMBER_PATTERN.search(question))
    return has_trigger and has_number


def parse_lookup(question: str):
    """Returns (dataset_tag_or_None, record_id_or_None)."""
    dataset = detect_dataset(question)
    match = NUMBER_PATTERN.search(question)
    record_id = int(match.group(1)) if match else None
    return dataset, record_id


## 4. Chunking (Reference #2 + #3)\n\nOnly applied to EDA report text -- real dataset rows from Section 2 are small enough to index unchunked.

In [ ]:
"""
chunking.py
-----------
Reference #2 (unstructured data) + reference #3 (large datasets): report
text mixes dense prose, table-as-text, and figure captions. A layered
recursive splitter keeps table rows attached to the sentence explaining
them instead of cutting mid-row. Overlap ensures facts sitting on a chunk
boundary still appear whole in at least one chunk.

Every chunk gets a stable content-hash ID scoped to its dataset tag, so
re-running ingestion on updated files doesn't create duplicate vectors.
"""

import hashlib
from langchain.text_splitter import RecursiveCharacterTextSplitter


def chunk_documents(docs, chunk_size: int = 900, chunk_overlap: int = 150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(docs)

    for c in chunks:
        h = hashlib.sha1(c.page_content.encode("utf-8")).hexdigest()[:10]
        c.metadata["chunk_id"] = f"{c.metadata.get('dataset', 'unk')}-{h}"

    per_dataset = {}
    for c in chunks:
        per_dataset[c.metadata.get("dataset", "unk")] = per_dataset.get(c.metadata.get("dataset", "unk"), 0) + 1
    print(f"[chunked] {len(docs)} docs -> {len(chunks)} chunks")
    for ds, n in per_dataset.items():
        print(f"    {ds}: {n} chunks")
    return chunks


## 5. Multimodal encoders -- BERT (text) + CLIP (image)\n\nImage encoding only fires where `has_image=True` (GIMMICK, DOSA, DRISHTIKON); other chunks get an all-zero placeholder.

In [ ]:
"""
encoders.py
-----------
Matches Fig. 1 / Section IV-C(4) of the CulturalQA paper: text encoded via
BERT/RoBERTa, images via CLIP/ViT. Generalized here to run across all 7
datasets -- image encoding only actually fires for chunks tagged
has_image=True by ingestion.py (GIMMICK, DOSA, DRISHTIKON); the other four
datasets get a fixed learned "no-image" placeholder vector instead of a
real CLIP embedding, so every chunk still has a consistent (text_emb,
image_emb) pair going into the MoE fusion layer.
"""

import torch
from transformers import AutoTokenizer, AutoModel, CLIPModel, CLIPProcessor
from PIL import Image
import requests
from io import BytesIO

TEXT_MODEL_NAME = "bert-base-uncased"      # swap for "roberta-base" if preferred
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class TextEncoder:
    def __init__(self, model_name: str = TEXT_MODEL_NAME):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()

    @torch.no_grad()
    def encode(self, texts, max_length: int = 256):
        if isinstance(texts, str):
            texts = [texts]
        inputs = self.tokenizer(
            texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt"
        ).to(DEVICE)
        out = self.model(**inputs)
        # CLS-token pooled representation (standard for BERT-family encoders)
        cls_emb = out.last_hidden_state[:, 0, :]
        return cls_emb.cpu()  # (batch, 768)


class ImageEncoder:
    def __init__(self, model_name: str = CLIP_MODEL_NAME):
        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.model = CLIPModel.from_pretrained(model_name).to(DEVICE).eval()
        self.embed_dim = self.model.config.projection_dim  # 512 for ViT-B/32

        # Fixed learned placeholder for datasets/chunks with no image
        # (SCALE, DIWALI, Durga QA, Indian Cultural Dataset, and any
        # text-only rows within GIMMICK/DOSA/DRISHTIKON). Zeros are used
        # rather than random noise so the MoE gate can learn to treat an
        # all-zero image vector as "no visual signal" cleanly.
        self.null_image_embedding = torch.zeros(1, self.embed_dim)

    def _load_image(self, ref: str):
        try:
            if ref.startswith("http"):
                resp = requests.get(ref, timeout=5)
                return Image.open(BytesIO(resp.content)).convert("RGB")
            return Image.open(ref).convert("RGB")
        except Exception as e:
            print(f"[image load failed] {ref}: {e}")
            return None

    @torch.no_grad()
    def encode(self, image_ref: str = None):
        if not image_ref:
            return self.null_image_embedding
        img = self._load_image(image_ref)
        if img is None:
            return self.null_image_embedding
        inputs = self.processor(images=img, return_tensors="pt").to(DEVICE)
        feats = self.model.get_image_features(**inputs)
        return feats.cpu()  # (1, 512)


class MultimodalEncoder:
    """Wraps both encoders and produces one (text_emb, image_emb) pair per
    chunk, respecting each dataset's HAS_IMAGE_MODALITY flag from
    ingestion.py."""

    def __init__(self):
        self.text_encoder = TextEncoder()
        self.image_encoder = ImageEncoder()

    def encode_chunk(self, chunk):
        text_emb = self.text_encoder.encode(chunk.page_content)[0]  # (768,)

        image_ref = None
        if chunk.metadata.get("has_image"):
            image_ref = (
                chunk.metadata.get("image_url")
                or chunk.metadata.get("image_path")
                or chunk.metadata.get("images")
            )
        image_emb = self.image_encoder.encode(image_ref)[0]  # (512,) or null vector

        return text_emb, image_emb


## 6. MoE Fusion Layer

Mixtral-style sparse top-k gating + DeepSeekMoE-style shared/routed expert split.

**Note:** randomly initialized -- architecturally complete, not yet trained. Used as an
auxiliary re-rank signal on top of the cross-encoder, not a standalone ranker.

In [ ]:
"""
moe_fusion.py
-------------
The fusion layer from Fig. 1 / Section IV-C(5): "A gating mechanism
dynamically routes information through specialized expert subnetworks."

Design pulls directly from the two MoE papers your related-work section
cites:
  - Mixtral [4]: sparse top-k gating -- only the top_k highest-scoring
    experts run per input, not all of them, keeping compute cheap even as
    more datasets/experts are added.
  - DeepSeekMoE [5]: splits experts into many small "routed" experts (finer
    specialization than a few big ones) and isolates a "shared" expert that
    always runs, so common cross-dataset patterns don't have to be
    re-learned redundantly by every routed expert.

Input: concatenated (text_emb, image_emb) per chunk -- image_emb is the
all-zero placeholder for the 4 text-only datasets, so the gate naturally
learns to route those toward experts that don't rely on visual features.

IMPORTANT: gate + expert weights are randomly initialized here. This
module defines the *architecture* the paper specifies; to make the
routing meaningful (rather than arbitrary) it needs training on a
relevance-labeled set of (query, chunk, is_relevant) triples per dataset,
using contrastive or triplet loss, before it should be trusted to
influence real ranking decisions. Until trained, treat its output as an
auxiliary score alongside the cross-encoder reranker, not a replacement.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

TEXT_DIM = 768
IMAGE_DIM = 512
FUSED_DIM = 256

NUM_ROUTED_EXPERTS = 8
TOP_K = 2


class Expert(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden: int = 512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class MoEFusion(nn.Module):
    def __init__(
        self,
        text_dim: int = TEXT_DIM,
        image_dim: int = IMAGE_DIM,
        fused_dim: int = FUSED_DIM,
        num_routed_experts: int = NUM_ROUTED_EXPERTS,
        top_k: int = TOP_K,
    ):
        super().__init__()
        self.top_k = top_k
        self.num_routed_experts = num_routed_experts

        in_dim = text_dim + image_dim

        # project raw concatenated multimodal features down before routing
        self.input_proj = nn.Linear(in_dim, fused_dim)

        # DeepSeekMoE-style split: many small routed experts + one always-on
        # shared expert that captures cross-dataset commonalities.
        self.routed_experts = nn.ModuleList(
            [Expert(fused_dim, fused_dim) for _ in range(num_routed_experts)]
        )
        self.shared_expert = Expert(fused_dim, fused_dim)

        # gating network: scores each routed expert given the projected input
        self.gate = nn.Linear(fused_dim, num_routed_experts)

    def forward(self, text_emb: torch.Tensor, image_emb: torch.Tensor):
        """
        text_emb:  (batch, TEXT_DIM)
        image_emb: (batch, IMAGE_DIM)  -- all-zero rows are valid ("no image")
        returns:   (batch, FUSED_DIM) fused representation
        """
        x = torch.cat([text_emb, image_emb], dim=-1)
        x = self.input_proj(x)

        gate_logits = self.gate(x)                      # (batch, num_routed_experts)
        topk_vals, topk_idx = torch.topk(gate_logits, self.top_k, dim=-1)
        topk_weights = F.softmax(topk_vals, dim=-1)      # normalize only over the chosen experts

        batch_size = x.size(0)
        routed_out = torch.zeros(batch_size, x.size(-1), device=x.device)

        for slot in range(self.top_k):
            expert_idx_per_row = topk_idx[:, slot]        # which expert each row picked at this slot
            weight_per_row = topk_weights[:, slot].unsqueeze(-1)
            for e in range(self.num_routed_experts):
                mask = (expert_idx_per_row == e)
                if mask.any():
                    routed_out[mask] += weight_per_row[mask] * self.routed_experts[e](x[mask])

        fused = self.shared_expert(x) + routed_out
        return fused

    @torch.no_grad()
    def score_relevance(self, query_text_emb, query_image_emb, chunk_text_emb, chunk_image_emb):
        """Auxiliary relevance score: cosine similarity between the fused
        query representation and the fused chunk representation. Used as a
        secondary signal alongside the cross-encoder reranker (see
        retrieval.py) -- not a standalone ranker until trained."""
        q_fused = self.forward(query_text_emb, query_image_emb)
        c_fused = self.forward(chunk_text_emb, chunk_image_emb)
        return F.cosine_similarity(q_fused, c_fused, dim=-1)


## 7. Retrieval -- FAISS + BM25 hybrid, query expansion, cross-encoder rerank (Reference #3)

In [ ]:
"""
retrieval.py
------------
Matches Fig. 1's "FAISS Retrieval" + "Cross-Encoder Reranker" stages, and
layers in the large-dataset strategies from reference #3:
  - Hybrid dense (FAISS) + sparse (BM25) retrieval via EnsembleRetriever,
    so exact numbers/proper nouns (BM25) and semantic matches (dense) both
    get caught -- matters more as 7 datasets' worth of chunks pile up.
  - MultiQueryRetriever expands a vague question into several phrasings
    before searching, improving recall on a large/varied corpus.
  - CrossEncoderReranker narrows the pooled candidates back down to a
    small, high-precision set before anything expensive happens downstream.
"""

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_groq import ChatGroq

INDEX_DIR = "./faiss_store"


def build_faiss_index(chunks, index_dir: str = INDEX_DIR):
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
    vectordb = FAISS.from_documents(chunks, embeddings)
    vectordb.save_local(index_dir)
    print(f"[faiss] index saved to {index_dir}")
    return vectordb


def load_faiss_index(index_dir: str = INDEX_DIR):
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
    return FAISS.load_local(index_dir, embeddings, allow_dangerous_deserialization=True)


def build_hybrid_retriever(vectordb, chunks, k: int = 8, dataset_filter: str = None):
    dense_kwargs = {"k": k}
    if dataset_filter:
        dense_kwargs["filter"] = {"dataset": dataset_filter}
    dense_retriever = vectordb.as_retriever(search_kwargs=dense_kwargs)

    bm25 = BM25Retriever.from_documents(chunks)
    bm25.k = k

    return EnsembleRetriever(retrievers=[dense_retriever, bm25], weights=[0.6, 0.4])


def build_query_expansion_retriever(base_retriever, expansion_model: str = "llama-3.1-8b-instant"):
    llm = ChatGroq(model=expansion_model, temperature=0)
    return MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)


def build_reranking_retriever(expanded_retriever, top_n: int = 6):
    cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
    reranker = CrossEncoderReranker(model=cross_encoder, top_n=top_n)
    return ContextualCompressionRetriever(base_compressor=reranker, base_retriever=expanded_retriever)


def build_full_retriever(vectordb, chunks, dataset_filter: str = None):
    """Full stack: hybrid -> query expansion -> cross-encoder rerank.
    This is what feeds the MoE fusion + extractive reader downstream."""
    hybrid = build_hybrid_retriever(vectordb, chunks, dataset_filter=dataset_filter)
    expanded = build_query_expansion_retriever(hybrid)
    reranked = build_reranking_retriever(expanded)
    return reranked


## 8. Extractive QA Reader

Pulls an exact answer span from retrieved text instead of generating a new sentence --
avoids the hallucination risk flagged in the paper's critique of generative decoders.

In [ ]:
"""
reader.py
---------
Matches Fig. 1's final stage: "Extractive QA Reader ... pulls answer spans
directly from retrieved context." Deliberately NOT a generative LLM call --
per your paper's own critique of MuRAG/Mixtral/DeepSeekMoE (Section II),
generative decoders risk producing answers that aren't grounded in any
retrieved source, which is exactly the failure mode this architecture is
built to avoid for underrepresented cultural facts.

Uses a RoBERTa model fine-tuned on SQuAD2 (deepset/roberta-base-squad2),
which also supports "no answer found" -- useful when none of the retrieved
chunks (across any of the 7 datasets) actually contain the answer, so the
system can say so instead of forcing an extraction.
"""

from transformers import pipeline

READER_MODEL = "deepset/roberta-base-squad2"


class ExtractiveReader:
    def __init__(self, model_name: str = READER_MODEL):
        self.qa_pipeline = pipeline("question-answering", model=model_name, tokenizer=model_name)

    def answer(self, question: str, candidate_chunks, min_score: float = 0.05):
        """Runs extraction over each candidate chunk independently (SQuAD-style
        readers score per-context) and returns the single best span across
        all candidates, tagged with which dataset it came from."""
        best = None
        for chunk in candidate_chunks:
            context = chunk.page_content
            if not context.strip():
                continue
            try:
                result = self.qa_pipeline(question=question, context=context)
            except Exception as e:
                print(f"[reader error] {e}")
                continue

            result["dataset"] = chunk.metadata.get("dataset")
            result["source_file"] = chunk.metadata.get("source_file")
            result["context"] = context

            if best is None or result["score"] > best["score"]:
                best = result

        if best is None or best["score"] < min_score:
            return {
                "answer": None,
                "message": "No confident answer span found in retrieved context across any dataset.",
            }
        return best


## 9. Full pipeline -- both paths wired together

In [ ]:
"""
pipeline.py
-----------
End-to-end wiring, mirroring Fig. 1 top to bottom -- now with two answer
paths instead of one:

  PATH A (exact lookup): "what's artifact #412 in DOSA?"
    -> query_router detects this is an ID question
    -> structured_store returns the exact row directly, no search needed
    -> deterministic, 100% accurate (if the record exists)

  PATH B (everything else): "how many artifacts does DOSA have?" /
  "which artifacts relate to weddings?"
    -> normal flow: FAISS+BM25 hybrid -> MultiQuery expansion ->
       Cross-Encoder rerank -> MoE re-score -> Extractive reader
    -> now searches BOTH the report chunks AND the real per-row records
       (structured_store.records_to_documents), so it can also surface
       actual dataset content, not just report descriptions

Runs across all 7 datasets at once (dataset_filter=None), or restrict to
one the way the original paper restricts to DOSA alone.
"""

import os
import json
import torch

# defined above in this notebook








CHUNK_CACHE = os.path.join(INDEX_DIR, "chunks_cache.json")
RAW_RECORDS_CACHE = os.path.join(INDEX_DIR, "raw_records_cache.json")


def build_index(data_dir: str = "./data"):
    # 1. descriptive EDA reports -> chunked (conceptual/statistical questions)
    report_docs = load_all_documents(data_dir)
    chunks = chunk_documents(report_docs) if report_docs else []

    # 2. real per-row dataset records -> NOT chunked (rows are small already),
    #    enables both exact lookup AND semantic search over actual content
    raw_records = load_all_raw_records(os.path.join(data_dir, "raw"))
    raw_docs = records_to_documents(raw_records)

    all_index_docs = chunks + raw_docs
    if not all_index_docs:
        raise SystemExit(
            "Nothing to index -- add EDA report files to ./data and/or "
            "real dataset files to ./data/raw/<Dataset>/."
        )

    build_faiss_index(all_index_docs)

    os.makedirs(INDEX_DIR, exist_ok=True)
    with open(CHUNK_CACHE, "w") as f:
        json.dump([{"content": d.page_content, "metadata": d.metadata} for d in all_index_docs], f)
    with open(RAW_RECORDS_CACHE, "w") as f:
        json.dump(raw_records, f)

    print(f"[done] index built: {len(chunks)} report chunks + {len(raw_docs)} raw records "
          f"= {len(all_index_docs)} total, across all 7 datasets.")


def _load_chunks_cache():
    from langchain.schema import Document
    with open(CHUNK_CACHE) as f:
        raw = json.load(f)
    return [Document(page_content=r["content"], metadata=r["metadata"]) for r in raw]


def _load_raw_records_cache():
    if not os.path.exists(RAW_RECORDS_CACHE):
        return []
    with open(RAW_RECORDS_CACHE) as f:
        return json.load(f)


class CulturalQAPipeline:
    def __init__(self, dataset_filter: str = None):
        self.vectordb = load_faiss_index()
        self.chunks = _load_chunks_cache()
        self.retriever = build_full_retriever(self.vectordb, self.chunks, dataset_filter=dataset_filter)
        self.encoder = MultimodalEncoder()
        self.moe = MoEFusion().eval()
        self.reader = ExtractiveReader()

        raw_records = _load_raw_records_cache()
        self.record_index = build_record_index(raw_records)
        print(f"[pipeline] {len(self.record_index)} raw records available for exact lookup "
              f"across datasets: {sorted(set(ds for ds, _ in self.record_index))}")

    # -------------------------------------------------------------- Path A
    def _try_exact_lookup(self, question: str):
        if not is_lookup_query(question):
            return None

        dataset, record_id = parse_lookup(question)
        if record_id is None:
            return None

        if dataset:
            record = self.record_index.get((dataset, record_id))
            if record:
                return {
                    "answer": format_record_answer(record),
                    "path": "exact_lookup",
                    "dataset": dataset,
                    "record_id": record_id,
                    "score": 1.0,
                }
            ids_for_dataset = sorted(rid for ds, rid in self.record_index if ds == dataset)
            if not ids_for_dataset:
                return {
                    "answer": None,
                    "message": f"No raw records loaded for {dataset} yet -- add its real data "
                               f"files under ./data/raw/{dataset}/ and rebuild the index.",
                }
            return {
                "answer": None,
                "message": f"No record #{record_id} found in {dataset}. "
                           f"Valid range is #{ids_for_dataset[0]}\u2013#{ids_for_dataset[-1]}.",
            }

        # no dataset named in the question -- check every dataset for that id
        matches = {ds: self.record_index[(ds, record_id)]
                   for (ds, rid) in self.record_index if rid == record_id}
        if not matches:
            return {"answer": None, "message": f"No record #{record_id} found in any loaded dataset. "
                                                f"Try naming the dataset, e.g. 'artifact #{record_id} in DOSA'."}
        if len(matches) == 1:
            ds, record = next(iter(matches.items()))
            return {"answer": format_record_answer(record), "path": "exact_lookup",
                    "dataset": ds, "record_id": record_id, "score": 1.0}
        return {
            "answer": None,
            "message": f"Record #{record_id} exists in multiple datasets ({', '.join(matches)}) -- "
                       f"specify which one, e.g. 'artifact #{record_id} in {list(matches)[0]}'.",
        }

    # -------------------------------------------------------------- Path B
    def _moe_rescore(self, question: str, candidates):
        q_text_emb = self.encoder.text_encoder.encode(question)
        q_image_emb = torch.zeros(1, self.encoder.image_encoder.embed_dim)

        scored = []
        for chunk in candidates:
            c_text_emb, c_image_emb = self.encoder.encode_chunk(chunk)
            moe_score = self.moe.score_relevance(
                q_text_emb, q_image_emb, c_text_emb.unsqueeze(0), c_image_emb.unsqueeze(0)
            ).item()
            scored.append((moe_score, chunk))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [c for _, c in scored]

    def _rag_answer(self, question: str, use_moe_rescore: bool = True, top_k_for_reader: int = 3):
        candidates = self.retriever.invoke(question)
        if use_moe_rescore and candidates:
            candidates = self._moe_rescore(question, candidates)

        top_candidates = candidates[:top_k_for_reader]
        result = self.reader.answer(question, top_candidates)
        result["path"] = "rag"
        result["retrieved_from"] = [
            {"dataset": c.metadata.get("dataset"), "source_file": c.metadata.get("source_file"),
             "source_type": c.metadata.get("source_type", "report_chunk")}
            for c in top_candidates
        ]
        return result

    # -------------------------------------------------------------- Entry point
    def ask(self, question: str, **rag_kwargs):
        lookup_result = self._try_exact_lookup(question)
        if lookup_result is not None:
            return lookup_result
        return self._rag_answer(question, **rag_kwargs)


def ask(question: str, dataset_filter: str = None):
    pipeline = CulturalQAPipeline(dataset_filter=dataset_filter)
    result = pipeline.ask(question)

    print("\n--- ANSWER ---")
    print(result.get("answer") or result.get("message"))
    if result.get("path") == "exact_lookup":
        print(f"(direct lookup, dataset={result['dataset']}, record #{result['record_id']})")
    elif result.get("answer"):
        print(f"(confidence: {result['score']:.3f}, from: {result.get('dataset')} / {result.get('source_file')})")
    if result.get("retrieved_from"):
        print("\n--- CANDIDATES CONSIDERED ---")
        for r in result["retrieved_from"]:
            print(f"- {r['dataset']} ({r.get('source_file') or r.get('source_type')})")
    return result


## 10. Build the index

Run once (or whenever `./data` / `./data/raw` changes). Loads report PDFs + real
dataset rows, builds the FAISS index, and caches the exact-lookup record index.

In [ ]:
build_index(data_dir="./data")

## 11. Ask questions -- try both paths

`dataset_filter=None` searches across all 7. Try an exact-lookup question and a
conceptual question back to back to see both paths fire.

In [ ]:
pipeline = CulturalQAPipeline(dataset_filter=None)

In [ ]:
# Path A: exact lookup (deterministic, no search)
result = pipeline.ask("what is artifact #3 in DOSA")
print(result.get("answer") or result.get("message"))
print("path:", result.get("path"))

In [ ]:
# Path B: RAG over report + real records
result = pipeline.ask("how many artifacts does DOSA have")
print(result.get("answer") or result.get("message"))
print("path:", result.get("path"))
for r in result.get("retrieved_from", []):
    print("  from:", r["dataset"], "-", r.get("source_file") or r.get("source_type"))

### Interactive loop (optional)

In [ ]:
while True:
    q = input("Ask> ").strip()
    if q.lower() in ("exit", "quit"):
        break
    result = pipeline.ask(q)
    print(result.get("answer") or result.get("message"))
    if result.get("path") == "exact_lookup":
        print(f"  (direct lookup: {result['dataset']} #{result['record_id']})")
    for r in result.get("retrieved_from", []):
        print("  from:", r["dataset"], "-", r.get("source_file") or r.get("source_type"))

## 12. Getting the remaining datasets in

**DIWALI, DRISHTIKON, Indian Cultural Dataset — no login needed, confirmed ungated:**
```python
from datasets import load_dataset
load_dataset("nlip/DIWALI")["train"].to_csv("./data/raw/DIWALI/data.csv")
load_dataset("13ari/DRISHTIKON")["train"].to_csv("./data/raw/DRISHTIKON/data.csv")
load_dataset("ombhojane/indian-cultural-dataset")["train"].to_json("./data/raw/Indian Cultural Dataset/data.json")
```
Run this locally (your machine has normal HuggingFace access, no gating) then either
upload the resulting files to me or drop them straight into the folders above --
`structured_store.py` already has real, schema-verified loaders for all three.

**GIMMICK — needs HF login (it's gated):**
```python
# after `huggingface-cli login` and accepting the license on the dataset page
from datasets import load_dataset
load_dataset("floschne/gimmick-civqa")["train"].to_csv("./data/raw/GIMMICK/civqa.csv")
load_dataset("floschne/gimmick-qa-base")["train"].to_csv("./data/raw/GIMMICK/qa_base.csv")
```

**Durga QA — not on HuggingFace.** It's on Mendeley Data
(DOI `10.17632/6hcx53kywr.1`) — download manually, then drop the CSV into
`./data/raw/Durga QA/`.

Once real files land in `./data/raw/<Dataset>/`, `structured_store.py`'s
`RAW_LOADERS` dict picks the right loader automatically for DOSA, SCALE, DIWALI,
DRISHTIKON, and Indian Cultural Dataset. For GIMMICK and Durga QA it falls back to
the generic loader — check the column names it infers look right before trusting
exact-lookup answers from those two.
